# Credit Risk & IFRS 9 Walkthrough

This notebook mirrors the recruiter-facing workflow of the repository:

1. generate a synthetic multi-period retail loan book,
2. fit a discrete-time survival PD model,
3. score one reporting date,
4. run the IFRS 9 ECL engine,
5. inspect monitoring and validation-style outputs.

References:
- Singer & Willett (1993): https://journals.sagepub.com/doi/10.3102/10769986018002155
- IFRS 9: https://www.ifrs.org/content/dam/ifrs/publications/pdf-standards/english/2021/issued/part-a/ifrs-9-financial-instruments.pdf
- SR 11-7: https://www.federalreserve.gov/boarddocs/srletters/2011/sr1107a1.pdf


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from credit_risk_lab import (
    PortfolioConfig,
    build_portfolio_report,
    fit_survival_pd_model,
    generate_portfolio_timeseries,
    run_ifrs9_pipeline,
    run_monitoring,
    score_portfolio,
)


In [ ]:
dataset = generate_portfolio_timeseries(
    PortfolioConfig(random_seed=11, periods=12, num_borrowers=160, num_loans=280)
)
dataset.performance.head()


In [ ]:
model = fit_survival_pd_model(dataset.performance)
model.coefficient_table.head(10)


In [ ]:
as_of_date = dataset.performance['snapshot_date'].max()
scores = score_portfolio(dataset, model, as_of_date)
ecl_report = run_ifrs9_pipeline(dataset, scores)
monitoring = run_monitoring(
    reference_snapshot=scores.previous_snapshot_scores,
    current_snapshot=scores.snapshot_scores,
    reference_scores=scores.previous_snapshot_scores,
    current_scores=scores.snapshot_scores,
)
ecl_report.stage_summary


In [ ]:
print(build_portfolio_report(dataset, scores, ecl_report, monitoring))
